# 01 — Fetch and explore the feed

What is actually in LA Metro's rail GTFS, before any of it is schematised.
The feed is cached under `data/feeds/`; re-running is free.

In [ ]:
%load_ext autoreload
%autoreload 2

from schematic import feeds, loom, pipeline, animate
from schematic.linegraph import LineGraph
from schematic.crs import to_mercator
from schematic.render import render, octilinearity, Style

FEED = "la-metro-rail"
LINE_ORDER = list("ABCDEK")   # the order lines are drawn in, back to front

In [ ]:
zip_path = feeds.fetch(FEED)
tables = feeds.tables(FEED)
print(zip_path.name, f"{zip_path.stat().st_size/1e6:.1f} MB")
{name: len(df) for name, df in sorted(tables.items())}

### Routes

LA leaves `route_short_name` blank and names its routes "Metro A Line", which
would give LOOM six lines all labelled `""`. `feeds.normalize()` fills the
column in before the feed ever reaches LOOM — and `feeds.tables()` reads that
normalised copy, so the labels here are the ones the map will use.

In [ ]:
tables["routes"][["route_id", "route_short_name", "route_long_name",
                  "route_type", "route_color"]]

### Stops

Trips call at platforms (`location_type=0`); each belongs to a parent station
(`location_type=1`), and the rest are entrances. Only the platforms matter for
matching the schedule to the map.

In [ ]:
stops = tables["stops"]
print("by location_type:", stops["location_type"].fillna("0").value_counts().to_dict())
used = set(tables["stop_times"]["stop_id"])
print("stop_ids actually served:", len(used))
stops[stops["stop_id"].isin(used)].head()

### Where the network actually is

In [ ]:
import matplotlib.pyplot as plt

pts = stops[stops["stop_id"].isin(used)]
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(pts["stop_lon"].astype(float), pts["stop_lat"].astype(float), s=8)
ax.set_aspect(1 / 0.83)   # rough cos(latitude) at LA
ax.set_title("LA Metro rail stops, geographic")
plt.show()